# Environment

In [1]:
import os
os.chdir("..")

In [ ]:
import pandas as pd
import spacy

from src.utils.path import DATA_PATH

In [4]:
nlp = spacy.load("pt_core_news_lg", disable=["tagger", "parser", "ner", "lemmatizer"])
stopwords = nlp.Defaults.stop_words

# Load Data

In [6]:
lula = pd.read_csv(f"{DATA_PATH}/ited-br/raw/lula.csv")
bolsonaro = pd.read_csv(f"{DATA_PATH}/ited-br/raw/bolsonaro.csv")

lula["candidate"] = "lula"
bolsonaro["candidate"] = "bolsonaro"

df = pd.concat([lula, bolsonaro], ignore_index=True)

# Analysis

## Data Overview

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 600002 entries, 0 to 600001
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   id         600002 non-null  int64
 1   text       600002 non-null  str  
 2   candidate  600002 non-null  str  
dtypes: int64(1), str(2)
memory usage: 112.0 MB


In [8]:
df.head()

,id,text,candidate
0,1584657183316512769,mas é o Lula que vai acabar com as igrejas kkk...,lula
1,1580889035949502465,@AdailtonMike @monicabergamo Nenhum momento af...,lula
2,1541202977100369920,.. esconder é fácil \n.. difícil é deixar de s...,lula
3,1582335548684836865,@PesqBR2022 simplesmente o cara que sabia que ...,lula
4,1579675165201666055,@Estadao @Miltonneves @EstadaoEconomia Simone ...,lula


## Text Length Analysis

In [9]:
df["char_count"] = df["text"].fillna("").str.len()
df["word_count"] = df["text"].fillna("").str.split().apply(len)

In [10]:
df.groupby("candidate")[["char_count", "word_count"]].describe()

char_count                                                          \
               count        mean        std   min   25%    50%    75%    max   
candidate                                                                      
bolsonaro   300000.0  162.419627  79.415286  16.0  96.0  142.0  228.0  991.0   
lula        300002.0  156.393941  83.106235  15.0  89.0  133.0  218.0  991.0   

          word_count                                                     
               count       mean        std  min   25%   50%   75%   max  
candidate                                                                
bolsonaro   300000.0  25.800157  13.395107  2.0  14.0  22.0  36.0  98.0  
lula        300002.0  25.353574  13.733079  1.0  14.0  21.0  36.0  99.0

## Clean Lexical Diversity (No Stopwords)

In [11]:
def get_clean_unique_words(text):
    tokens = str(text).lower().split()
    return len(set(w for w in tokens if w not in stopwords and w.isalpha()))

In [12]:
df["clean_unique_words"] = df["text"].fillna("").apply(get_clean_unique_words)
df["lexical_diversity"] = df["clean_unique_words"] / df["word_count"].replace(0, 1)

In [13]:
df.groupby("candidate")[["clean_unique_words", "lexical_diversity"]].describe()

clean_unique_words                                                 \
                       count      mean       std  min  25%  50%   75%   max   
candidate                                                                     
bolsonaro           300000.0  8.837760  4.941775  0.0  5.0  8.0  12.0  39.0   
lula                300002.0  8.640552  4.978040  0.0  5.0  8.0  12.0  38.0   

          lexical_diversity                                               \
                      count      mean       std  min       25%       50%   
candidate                                                                  
bolsonaro          300000.0  0.350823  0.114872  0.0  0.277778  0.346154   
lula               300002.0  0.349882  0.116046  0.0  0.277778  0.345455   

                          
                75%  max  
candidate                 
bolsonaro  0.418182  1.0  
lula       0.416667  1.0

### Top Words Frequency (No Stopwords)

In [14]:
lula_tokens = " ".join(df[df["candidate"] == "lula"]["text"].fillna("").str.lower()).split()
lula_words = pd.Series([w for w in lula_tokens if w not in stopwords and w.isalpha()]).value_counts()
lula_words.head(20)

lula          224691
bolsonaro      47769
pra            39503
q              25630
presidente     22242
vc             18320
votar          16978
tá             16447
brasil         16376
voto           13883
pq             12248
ladrão         11627
pt             11003
governo        10932
dia             9960
gente           9884
vou             9879
pro             9783
turno           9571
cara            9468
Name: count, dtype: int64

In [ ]:
bolsonaro_tokens = " ".join(df[df["candidate"] == "bolsonaro"]["text"].fillna("").str.lower()).split()
bolsonaro_words = pd.Series([w for w in bolsonaro_tokens if w not in stopwords and w.isalpha()]).value_counts()
bolsonaro_words.head(20)

bolsonaro     186721
lula           52527
pra            37900
q              26107
jair           21515
bozo           19587
vc             17437
tá             17232
presidente     16983
brasil         14885
votar          14218
janones        13259
pq             12351
voto           12221
censurou       12034
governo        11935
anos           10315
gente           9972
cara            9971
dia             9420
Name: count, dtype: int64

: 